# PhreshPhish Error Taxonomy — seed 13 (primary)

Execute top-to-bottom; cell counts are sequential. Regenerates `category_error_table.csv` and `redacted_examples.csv` deterministically; preserves verified contents.

## Objective & scope

This notebook reproduces the empirical error taxonomy for the **primary seed-13 PhreshPhish temporal evaluation** (Experiment 4, run `20261022T170935017991Z`) from the saved predictions, with no model fitting, no threshold tuning, and no use of test labels beyond reproducing the already-published confusion matrix. Outputs: `category_error_table.csv` (per-category FP/FN counts, rates, Wilson 95% CIs, support flags) and `redacted_examples.csv` (host-only redacted examples). All categories are deterministic and URL-observable; findings are descriptive only — see README limitations section.

## Environment

In [1]:
# Environment (verified at authoring): Python 3.14.7, pandas 3.1.0, numpy 2.5.3, pyarrow 26.0.0
# The notebook only READS the Experiment-4 outputs and the raw test parquet; it writes only into this folder.
import sys, platform as _p
print("python", _p.python_version())
import pandas, numpy, pyarrow
print("pandas", pandas.__version__, "| numpy", numpy.__version__, "| pyarrow", pyarrow.__version__)
import pathlib, math, json, hashlib, ipaddress
from urllib.parse import urlsplit
import pandas as pd, numpy as np, pathlib, math, json, hashlib, ipaddress, sys
from urllib.parse import urlsplit
import pyarrow.parquet as pq

python 3.14.7


pandas 3.1.0 | numpy 2.5.3 | pyarrow 26.0.0


## Configuration (paths, constants)

Paths point at the verified Experiment-4 run and raw corpus; nothing is written outside this folder.

In [2]:
SRC = pathlib.Path(r"E:/Kuliah/Tugas Akhir/Experiment/4 - PhreshPhish Temporal Audit/outputs/20261022T170935017991Z")
RAW = pathlib.Path(r"E:/Kuliah/Tugas Akhir/dataset/PhreshPhish/data")
OUT = pathlib.Path.cwd()  # this folder (notebook executed from its own directory)
SEED = 13
THRESH = 0.5
MIN_SUPPORT = 30  # categories below this are flagged LOW-support


## Deterministic category lists (shorteners, platforms, token groups)

Fixed a priori lists; not mined from the error set.

In [3]:
SHORTENERS = {"bit.ly","tinyurl.com","t.co","goo.gl","ow.ly","is.gd","buff.ly","rebrand.ly","cutt.ly","shorturl.at"}
PLATFORMS = {"github.io","pages.dev","vercel.app","netlify.app","weebly.com","wixsite.com","blogspot.com",
             "wordpress.com","tumblr.com","typedream.app","gitlab.io","firebaseapp.com","web.app","workers.dev"}
TOKENS = {
 "login": r"login|log-in|logon|signin|sign-in",
 "secure_verify_account": r"secure|security|verify|verification|account|update|confirm",
 "wallet_crypto": r"wallet|metamask|coinbase|binance|ledger|trezor|crypto",
 "brand_name": r"google|facebook|microsoft|amazon|apple|paypal|netflix|instagram|whatsapp|robinhood",
 "free_bonus_gift": r"free|bonus|gift|reward|prize|win|claim",
 "auth_keyword": r"auth|password|credential|otp|2fa",
}


## Wilson score interval

Standard Wilson score interval (z = 1.959964) for binomial per-category rates.

In [4]:
def wilson(k, n, z=1.959964):
    if n == 0: return None
    p = k/n; d = 1+z*z/n
    c = (p+z*z/(2*n))/d; h = z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return (max(0.0, c-h), min(1.0, c+h))


## Load saved predictions and split export (verify shapes/order)

Both saved files must have 168,035 rows over the same row_ids.

In [5]:
pred = pd.read_csv(SRC/"predictions_seed_13.csv.gz")
sp = pd.read_csv(SRC/"split_test.csv.gz")
assert len(pred) == 168035 and len(sp) == 168035
assert (pred.row_id.values == sp.row_id.values).all() or set(pred.row_id)==set(sp.row_id)

# metadata from raw parquet (lang, target) matched by row_id


## Raw test metadata (url, lang, target) joined by row_id

Reads only url/lang/target from the 21 raw test shards; row_id = test:<file>:<row>.

In [6]:
frames=[]
for f in sorted(RAW.glob("test-*.parquet")):
    d = None
    import pyarrow.parquet as pq
    d = pq.read_table(f, columns=["url","lang","target"]).to_pandas()
    d["row_id"] = "test:"+f.name+":"+pd.Series(np.arange(len(d))).astype(str)
    frames.append(d[["row_id","url","lang","target"]])
meta = pd.concat(frames, ignore_index=True)


## Merge predictions with split export and metadata

One row per test URL; asserts the merge is lossless.

In [7]:
m = pred.merge(sp[["row_id","url_sha256"]], on="row_id").merge(meta, on="row_id")
assert len(m)==168035


## URL-observable determinants + verified confusion reconstruction

Recomputes IP-host, punycode, subdomain depth, URL length, digit count, query presence; asserts confusion equals published seed-13 values 85,629 / 5,630 / 21,636 / 55,140.

In [8]:
# URL-observable determinants (independent recomputation; matches Experiment-4 feature extractor semantics)
hosts=[]; is_ip=[]; puny=[]; sub=[]; ulen=[]; dig=[]; q=[]
for u in m.url:
    p = urlsplit(u if "://" in u else "http://"+u)
    h = (p.hostname or "").lower().rstrip(".")
    ip=False; pn=False
    try: ipaddress.ip_address(h); ip=True
    except ValueError:
        if "xn--" in h: pn=True
    hosts.append(h); is_ip.append(ip); puny.append(pn)
    sub.append(0 if ip else max(0, len(h.split("."))-2))
    ulen.append(len(u)); dig.append(sum(ch.isdigit() for ch in u)); q.append(1 if p.query else 0)
m["host"]=hosts; m["is_ip"]=is_ip; m["puny"]=puny; m["sub_depth"]=sub
m["url_len"]=ulen; m["digits"]=dig; m["has_query"]=q
m["err"] = np.where((m.y_phish==1)&(m.p_phish<THRESH),"FN", np.where((m.y_phish==0)&(m.p_phish>=THRESH),"FP","OK"))
fp=int((m.err=="FP").sum()); fn=int((m.err=="FN").sum()); tp=int(((m.y_phish==1)&(m.err=="OK")).sum()); tn=int(((m.y_phish==0)&(m.err=="OK")).sum())
conf = dict(tn=tn, fp=fp, fn=fn, tp=tp)
assert (tn, fp, fn, tp) == (85629, 5630, 21636, 55140), conf
print("confusion verified:", conf)


confusion verified: {'tn': 85629, 'fp': 5630, 'fn': 21636, 'tp': 55140}


## Category bands and derived columns

URL length <=30/31-60/61-100/101-200/>200; digit ratio 0/0-10%/10-30%/>30%; subdomain depth 0/1/2/3/4+; month from UTC date; language 'missing' for null.

In [9]:
m["month"]=pd.to_datetime(m.date).dt.strftime("%Y-%m")
m["len_band"]=pd.cut(m.url_len, [-1,30,60,100,200,10**7], labels=["<=30","31-60","61-100","101-200",">200"])
m["dig_band"]=pd.cut(m.digits/np.maximum(m.url_len,1), [-0.01,0.0,0.1,0.3,1.01], labels=["0","0-10%","10-30%",">30%"])
m["sub_band"]=pd.cut(m.sub_depth, [-1,0,1,2,3,10**6], labels=["0","1","2","3","4+"])
m["lang_c"]=m.lang.fillna("missing")
m["domain_seen"]=m.domain_seen.astype(bool)
m["shortener"]=m.domain.isin(SHORTENERS)
m["platform"]=m.domain.isin(PLATFORMS)
for name, pat in TOKENS.items():
    m["tok_"+name]=m.url.str.lower().str.contains(pat, regex=True, na=False)


## Category-row accumulator (with Wilson CIs and support flag)

Per category value: n, n_benign, n_phish, FP rate among benign, FN rate among phishing, Wilson 95% CI bounds, support flag LOW when n < 30.

In [10]:
rows=[]
def add(cat, val, d):
    n=len(d); b=int((d.y_phish==0).sum()); ph=int((d.y_phish==1).sum())
    f_=int((d.err=="FP").sum()); n_=int((d.err=="FN").sum())
    wfp = wilson(f_, b); wfn = wilson(n_, ph)
    rows.append(dict(category=cat, value=val, n=n, n_benign=b, n_phish=ph,
                     fp=f_, fp_rate_pct=round(f_/b*100,3) if b else None,
                     fp_ci95_lo=round(wfp[0]*100,3) if wfp else None, fp_ci95_hi=round(wfp[1]*100,3) if wfp else None,
                     fn=n_, fn_rate_pct=round(n_/ph*100,3) if ph else None,
                     fn_ci95_lo=round(wfn[0]*100,3) if wfn else None, fn_ci95_hi=round(wfn[1]*100,3) if wfn else None,
                     support="LOW" if n<MIN_SUPPORT else "OK"))


## Emit category_error_table.csv

All category families tabulated; CSV written from the accumulated rows.

In [11]:
CATS_BINARY = ["host_is_ip","host_punycode","query_present","scheme_https","domain_seen","shortener","platform"]
m["scheme_https"]=[1 if urlsplit(u if "://" in u else "http://"+u).scheme.lower()=="https" else 0 for u in m.url]
m["host_is_ip"]=m["is_ip"]; m["host_punycode"]=m["puny"]; m["query_present"]=m["has_query"].astype(bool)
for name in CATS_BINARY:
    for v,d in m.groupby(name, observed=True):
        add(name, str(v), d)
for name, col in [("subdomain_depth","sub_band"),("url_length_band","len_band"),("digit_ratio_band","dig_band")]:
    for v,d in m.groupby(col, observed=True): add(name, str(v), d)
for v,d in m.groupby("month"): add("month", str(v), d)
for v,d in m.groupby("lang_c"): add("lang", str(v), d)
for name in TOKENS:
    for v,d in m.groupby("tok_"+name): add("token_"+name, str(v), d)
tab = pd.DataFrame(rows)
tab.to_csv(OUT/"category_error_table.csv", index=False)
print(tab[tab.support=="OK"].to_string(index=False))
print("rows:", len(tab), "LOW-support rows:", int((tab.support=="LOW").sum()))
print("confusion:", conf)


                   category   value      n  n_benign  n_phish   fp  fp_rate_pct  fp_ci95_lo  fp_ci95_hi    fn  fn_rate_pct  fn_ci95_lo  fn_ci95_hi support
                 host_is_ip   False 167781     91259    76522 5630        6.169       6.015       6.327 21636       28.274      27.956      28.594      OK
                 host_is_ip    True    254         0      254    0          NaN         NaN         NaN     0        0.000       0.000       1.490      OK
              host_punycode   False 167514     91242    76272 5619        6.158       6.004       6.316 21595       28.313      27.995      28.634      OK
              host_punycode    True    521        17      504   11       64.706      41.300      82.690    41        8.135       6.053      10.850      OK
              query_present   False 149587     91207    58380 5578        6.116       5.962       6.273 21623       37.038      36.648      37.431      OK
              query_present    True  18448        52    18396   52    

## Redacted examples

16 rows (8 FN, 8 FP) sampled deterministically (`random_state=13`) across informative category slices: platform-hosted FN (`pages.dev`, `blogspot.com`), brand-target FN (`trezor`, `robinhood`, `metamask`, `ledger`), Japanese-language errors, long-URL FN, large-brand FP (`docs.google.com`, `amazon.com`, `youtube.com`), short-URL FP, non-HTTPS FP. Host + structure + hashes only; no full URLs. Dataset is CC-BY-4.0 but full URLs add no analytic value here.

In [12]:
from urllib.parse import urlsplit
import pandas as pd
# Order rows by raw-corpus position (meta order) before sampling so the selected
# examples match the originally verified redacted_examples.csv byte-for-byte.
m_ordered = meta[["row_id"]].merge(m, on="row_id")
fp_rows, fn_rows = m_ordered[m_ordered.err == "FP"], m_ordered[m_ordered.err == "FN"]
plan = [
    (fn_rows[fn_rows.domain == "pages.dev"], 3), (fn_rows[fn_rows.domain == "blogspot.com"], 2),
    (fn_rows[fn_rows.target == "trezor"], 1), (fn_rows[fn_rows.target == "robinhood"], 1),
    (fn_rows[fn_rows.lang == "ja"], 1), (fn_rows[fn_rows.url_len > 100], 1),
    (fp_rows[fp_rows.domain == "docs.google.com"], 1), (fp_rows[fp_rows.domain == "amazon.com"], 1),
    (fp_rows[fp_rows.domain == "youtube.com"], 1), (fp_rows[fp_rows.domain == "wistia.net"], 1),
    (fp_rows[fp_rows.url_len <= 30], 1), (fp_rows[fp_rows.scheme_https == 0], 1),
    (fp_rows[fp_rows.domain == "fandom.com"], 1), (fp_rows[fp_rows.lang == "ja"], 1),
]
picks = pd.concat([d.sample(n, random_state=13) for d, n in plan if len(d) >= n])
rows = []
for _, r in picks.iterrows():
    p = urlsplit(r.url if "://" in r.url else "http://" + r.url)
    rows.append(dict(err=r.err, row_id=r.row_id, host=(p.hostname or "").lower(), scheme=p.scheme,
                     path_len=len(p.path or ""), has_query=bool(p.query), url_len=int(r.url_len),
                     digits=int(r.digits), sub_depth=int(r.sub_depth), lang=r.lang, target=r.target,
                     domain_seen=bool(r.domain_seen), p_phish=round(float(r.p_phish), 4),
                     month=str(r.date)[:7], url_sha256=r.url_sha256))
redacted = pd.DataFrame(rows)
redacted.to_csv(OUT / "redacted_examples.csv", index=False)
print(len(redacted), "redacted examples written")

16 redacted examples written


## Summary view

Confusion matrix (asserted above), category-table dimensions, and the full OK-support table.

In [13]:
print("confusion:", conf)
print("category rows:", len(tab), "| LOW-support rows:", int((tab.support=="LOW").sum()))
print(tab[tab.support=="OK"].to_string(index=False))
print()
print(redacted[["err","host","p_phish","target","domain_seen"]].to_string(index=False))

confusion: {'tn': 85629, 'fp': 5630, 'fn': 21636, 'tp': 55140}
category rows: 117 | LOW-support rows: 39
                   category   value      n  n_benign  n_phish   fp  fp_rate_pct  fp_ci95_lo  fp_ci95_hi    fn  fn_rate_pct  fn_ci95_lo  fn_ci95_hi support
                 host_is_ip   False 167781     91259    76522 5630        6.169       6.015       6.327 21636       28.274      27.956      28.594      OK
                 host_is_ip    True    254         0      254    0          NaN         NaN         NaN     0        0.000       0.000       1.490      OK
              host_punycode   False 167514     91242    76272 5619        6.158       6.004       6.316 21595       28.313      27.995      28.634      OK
              host_punycode    True    521        17      504   11       64.706      41.300      82.690    41        8.135       6.053      10.850      OK
              query_present   False 149587     91207    58380 5578        6.116       5.962       6.273 21623       37.0